In [1]:
!pip install timm

  Using cached pyyaml-6.0.3-cp310-cp310-win_amd64.whl (158 kB)


You should consider upgrading via the 'C:\AI\finalyrproject\diabetic-retinopathy\dr-backend\venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [23]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import timm
from tqdm.notebook import tqdm

In [24]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")

Using Device: cuda
GPU Device Name: NVIDIA GeForce RTX 3050 Laptop GPU


In [25]:
TRAIN_CSV_PATH = r"C:\AI\finalyrproject\diabetic-retinopathy\dr-backend\aptos\train_1.csv"
TRAIN_IMG_DIR = r"C:\AI\finalyrproject\diabetic-retinopathy\dr-backend\aptos\train_images"


In [26]:
df = pd.read_csv(TRAIN_CSV_PATH)
train_df, val_df = train_test_split(df, test_size=0.15, random_state=42, stratify=df['diagnosis'])

In [27]:
class APTOSDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = str(self.df.loc[idx, 'id_code']) + ".png"
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert("RGB")
        label = int(self.df.loc[idx, 'diagnosis'])

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)

In [28]:
# Standard ImageNet augmentations & normalization
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [29]:
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [30]:
train_dataset = APTOSDataset(train_df, TRAIN_IMG_DIR, transform=train_transform)
val_dataset = APTOSDataset(val_df, TRAIN_IMG_DIR, transform=val_transform)

In [31]:
# num_workers=0 ensures Windows stability inside Jupyter Notebooks
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=0, pin_memory=True)

In [32]:
print(f"Loaded {len(train_dataset)} training and {len(val_dataset)} validation rows.")

Loaded 2490 training and 440 validation rows.


In [33]:
model = timm.create_model(
    'swin_tiny_patch4_window7_224', 
    pretrained=True, 
    num_classes=5
)

In [34]:
model = model.to(DEVICE)

In [35]:
EPOCHS = 10
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()

In [36]:
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))

In [37]:
best_val_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    print(f"\n--- Starting Epoch {epoch+1}/{EPOCHS} ---", flush=True)

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]"):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()

        with torch.amp.autocast('cuda', enabled=(DEVICE.type == 'cuda')):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels.data)
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = (correct.double() / total).item()

    # Validation Pass
    model.eval()
    val_loss = 0.0
    val_preds, val_labels = [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            with torch.amp.autocast('cuda', enabled=(DEVICE.type == 'cuda')):
                outputs = model(images)
                loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)

            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    val_loss = val_loss / len(val_dataset)
    val_acc = accuracy_score(val_labels, val_preds)

    print(f"Epoch {epoch+1}/{EPOCHS} Summary:", flush=True)
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}%", flush=True)
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc*100:.2f}%", flush=True)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_swin_dr.pth")
        print("--> Saved best checkpoint: best_swin_dr.pth", flush=True)


--- Starting Epoch 1/10 ---


Epoch 1/10 [Train]:   0%|          | 0/156 [00:00<?, ?it/s]

Epoch 1/10 Summary:
Train Loss: 0.7314 | Train Acc: 72.69%
Val Loss:   0.5237 | Val Acc:   77.50%
--> Saved best checkpoint: best_swin_dr.pth

--- Starting Epoch 2/10 ---


Epoch 2/10 [Train]:   0%|          | 0/156 [00:00<?, ?it/s]

Epoch 2/10 Summary:
Train Loss: 0.5417 | Train Acc: 78.76%
Val Loss:   0.5917 | Val Acc:   75.00%

--- Starting Epoch 3/10 ---


...

Epoch 8/10 Summary:
Train Loss: 0.2896 | Train Acc: 88.76%
Val Loss:   0.3412 | Val Acc:   89.77%
--> Saved best checkpoint: best_swin_dr.pth

--- Starting Epoch 9/10 ---


Epoch 9/10 [Train]:   0%|          | 0/156 [00:00<?, ?it/s]

Epoch 9/10 Summary:
Train Loss: 0.2646 | Train Acc: 89.96%
Val Loss:   0.3319 | Val Acc:   90.68%
--> Saved best checkpoint: best_swin_dr.pth

--- Starting Epoch 10/10 ---


Epoch 10/10 [Train]:   0%|          | 0/156 [00:00<?, ?it/s]

Epoch 10/10 Summary:
Train Loss: 0.2467 | Train Acc: 90.76%
Val Loss:   0.3521 | Val Acc:   89.55%
